<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller: El Torneo del Colegio y los Apuntes Fotocopiados 🏆🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Taller Práctico Evaluativo · Módulo 01
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/homeworks/Para%20Dummies/01_Procesamiento_Almacenamiento_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 🎈 ¿Qué vamos a hacer en este taller?

En el colegio hay un **torneo de microfútbol** y las anotaciones se cuentan en **tres mesas** a la vez; además, los apuntes del curso se **fotocopian y se guardan en varias casas** por si una se cierra. Es la idea de procesar y guardar datos en **muchas máquinas**, en pequeño. Vas a:

1. 🗳️ **Contar como MapReduce:** cada mesa cuenta lo suyo y entrega un **acta** (el *map* con su *combiner*); el centro **suma las actas** (el *reduce*) y llegan menos hojas que papeletas.
2. 📚 **Cuidar copias como un mini-HDFS:** cada apunte vive en 3 casas; si **una casa cierra**, ¿se perdió algún apunte? ¿cuántas fotocopias hay que sacar para volver a tener 3?
3. ✍️ **Contar con tus palabras** por qué sirven las copias y las actas.

> 🧸 **Cómo funciona:** cada reto trae un cuadro de código con huecos `...` y pistas 💡. Reemplaza los `...` por lo que corresponda. Debajo hay una **autoverificación** que te orienta con cariño. No traen soluciones: ¡inténtalo tú!

### 📋 Instrucciones

- Completa solo los huecos `...`; no cambies las celdas de preparación ni de autoverificación.
- Al final usa *Restart & Run All*: todo debe correr sin errores y sin `...` pendientes.
- Todo funciona sin internet y sin *Spark* (datos inventados, escritos a mano). Repasa si quieres: [Contar los votos en varias mesas](../../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/Para%20Dummies/01_MapReduce_Dummies.ipynb) y [La biblioteca con copias en varias sedes](../../01%20-%20Procesamiento%20y%20Almacenamiento%20Distribuido/Para%20Dummies/02_Frameworks_Distribuidos_Dummies.ipynb).

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos inventados
# ============================================================
from collections import Counter

# Reto 1: las anotaciones del torneo, anotadas en tres mesas
mesas = {
    "Mesa 1": ["Halcones", "Tigres", "Halcones", "Cóndores", "Halcones", "Halcones", "Tigres", "Halcones"],
    "Mesa 2": ["Tigres", "Halcones", "Halcones", "Cóndores", "Cóndores", "Halcones", "Tigres", "Halcones", "Halcones", "Tigres"],
    "Mesa 3": ["Cóndores", "Halcones", "Halcones", "Tigres", "Halcones", "Halcones", "Halcones"],
}

# Reto 2: en qué casas está cada apunte (3 copias por apunte, 4 casas)
CASAS = ["Ana", "Beto", "Carla", "Diego"]
reparto = {
    "apunte1": ["Ana", "Beto", "Carla"],
    "apunte2": ["Beto", "Carla", "Diego"],
    "apunte3": ["Ana", "Carla", "Diego"],
    "apunte4": ["Ana", "Beto", "Diego"],
    "apunte5": ["Ana", "Beto", "Carla"],
    "apunte6": ["Beto", "Carla", "Diego"],
}
print("Papeletas por mesa:", {m: len(v) for m, v in mesas.items()}, "| apuntes:", len(reparto))

---
## Reto 1: Cada mesa cuenta lo suyo y manda un acta 🗳️

Si cada papeleta viajara sola hasta el centro, haría falta mucho papel. En cambio, cada **mesa** cuenta sus anotaciones y entrega un **acta** (un renglón por equipo: «Halcones: 5»). El centro **suma las actas**. Eso es MapReduce con *combiner*.

**Pasos**
1. `contar_acta(anotaciones)`: devuelve un diccionario `{equipo: cuántas veces aparece}`; con él arma `actas`, una por mesa.
2. `total`: el diccionario que resulta de **sumar las actas** (equipo por equipo).
3. `n_papeletas`: papeletas en total; `n_renglones`: renglones que se mandan al centro (la suma de los tamaños de las actas).

**Puntaje sugerido:** 35 puntos.

In [ ]:
# Casillas que debes llenar (empiezan vacías; no las borres)
actas = total = n_papeletas = n_renglones = None

# 💡 1) El acta de una mesa: contar cuántas veces sale cada equipo
def contar_acta(anotaciones):
    acta = {}
    for equipo in anotaciones:
        acta[equipo] = acta.get(equipo, 0) + ...     # suma 1 cada vez
    return acta

actas = [contar_acta(...) for m in mesas.values()]

# 💡 2) El centro suma las actas, equipo por equipo
total = {}
for acta in actas:
    for equipo, n in acta.items():
        total[equipo] = total.get(equipo, 0) + ...    # suma lo que dice el acta

# 💡 3) ¿Cuántas papeletas y cuántos renglones de acta?
n_papeletas = sum(len(m) for m in mesas.values())
n_renglones = sum(len(...) for acta in actas)

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert actas is not None and len(actas) == 3 and actas[0] == {"Halcones": 5, "Tigres": 2, "Cóndores": 1}, "Pista 🧸: actas tiene un acta por mesa: contar_acta(m) para cada m en mesas.values(). La Mesa 1 debe dar Halcones 5, Tigres 2, Cóndores 1."
assert contar_acta(["a", "b", "a"]) == {"a": 2, "b": 1}, "Pista 🧸: contar_acta(['a','b','a']) debe dar {'a': 2, 'b': 1}: suma 1 cada vez que aparece el equipo."
assert total is not None and total == dict(Counter(e for m in mesas.values() for e in m)), "Pista 🧸: total suma, equipo por equipo, lo que dice cada acta (total[equipo] = total.get(equipo, 0) + n)."
assert total == {"Halcones": 15, "Tigres": 6, "Cóndores": 4}, "El resultado final debe ser Halcones 15, Tigres 6 y Cóndores 4."
assert n_papeletas == 25 and n_renglones == 9, "Pista 🧸: hay 25 papeletas pero solo 9 renglones de acta (3 equipos × 3 mesas)."
assert n_renglones < n_papeletas, "Las actas deben mandar menos hojas que las papeletas sueltas: ¡ese es el ahorro del combiner!"
print(f"✅ ¡Reto 1 superado! {n_papeletas} papeletas se resumieron en {n_renglones} renglones de acta y el centro obtuvo {total}.")

---
## Reto 2: ¿Se perdió algún apunte si una casa cierra? 📚

Cada apunte está fotocopiado en **3 casas** de 4 (`reparto`). La casa de **Ana cierra** y sus copias desaparecen. En un mini-HDFS el «catálogo» se da cuenta y manda sacar fotocopias para volver a tener 3 copias de todo.

**Pasos**
1. `sin_ana`: el reparto **sin Ana** (cada apunte conserva solo las casas que no son Ana).
2. `perdidos`: lista de apuntes que quedaron **sin ninguna copia**.
3. `copias_faltantes`: cuántas copias hay que sacar para volver a 3 en cada apunte (suma de `3 - copias que quedaron`).
4. `reparado`: para cada apunte, añade casas de las que siguen abiertas (las de `CASAS` que no son Ana) hasta tener 3 copias.

**Puntaje sugerido:** 40 puntos.

In [ ]:
# Casillas que debes llenar
sin_ana = perdidos = copias_faltantes = reparado = None

# 💡 1) Quitar a Ana de cada lista
sin_ana = {apunte: [casa for casa in casas if casa != ...] for apunte, casas in reparto.items()}

# 💡 2) Apuntes cuya lista quedó vacía (¿alguno?)
perdidos = [apunte for apunte, casas in sin_ana.items() if len(casas) == ...]

# 💡 3) Fotocopias por sacar: lo que falta para llegar a 3 en cada apunte
copias_faltantes = sum(3 - len(...) for casas in sin_ana.values())

# 💡 4) Completar con casas abiertas hasta tener 3 copias
abiertas = [casa for casa in CASAS if casa != "Ana"]
reparado = {}
for apunte, casas in sin_ana.items():
    nuevas = list(casas)
    for casa in abiertas:
        if len(nuevas) < ... and casa not in nuevas:
            nuevas.append(casa)
    reparado[apunte] = nuevas

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert sin_ana is not None and all("Ana" not in v for v in sin_ana.values()) and len(sin_ana) == 6, "Pista 🧸: para cada apunte deja solo las casas distintas de 'Ana' (casa != 'Ana')."
assert sin_ana["apunte2"] == ["Beto", "Carla", "Diego"] and sin_ana["apunte1"] == ["Beto", "Carla"], "Revisa: el apunte2 no estaba en casa de Ana (queda igual) y el apunte1 queda con Beto y Carla."
assert perdidos == [], "Pista 🧸: ningún apunte queda sin copias, porque cada uno estaba en 3 casas y solo cerró 1. perdidos debe ser una lista vacía."
assert copias_faltantes == 4, "Pista 🧸: Ana guardaba 4 apuntes (1, 3, 4 y 5), así que hay que sacar 4 fotocopias."
assert reparado is not None and all(len(v) == 3 and len(set(v)) == 3 and "Ana" not in v for v in reparado.values()), "Pista 🧸: cada apunte debe quedar con 3 casas distintas y ninguna es Ana."
assert all(set(sin_ana[a]) <= set(reparado[a]) for a in reparado), "Las copias que sobrevivieron se conservan; solo se añaden casas nuevas."
print(f"✅ ¡Reto 2 superado! Con una casa cerrada se perdieron {len(perdidos)} apuntes y hubo que sacar {copias_faltantes} fotocopias para volver a 3 copias de cada uno.")

---
## Reto 3: ¿Por qué sirven las actas y las copias? ✍️

**Puntaje sugerido:** 25 puntos (sin código). Escribe **3 a 5 líneas** en la celda de abajo respondiendo:

- ¿Por qué mandar **9 renglones de acta** es mejor que mandar **25 papeletas** al centro? ¿Qué pasaría con 25 millones de papeletas?
- ¿Por qué no se perdió ningún apunte cuando cerró la casa de Ana? ¿Qué habría pasado si cada apunte tuviera **una sola copia**?
- ¿Para qué sirve «sacar fotocopias» después de la falla, si ya no se había perdido nada?

---
#### ✍️ Tu respuesta

_Escribe aquí tus 3-5 líneas. Reemplaza este texto por tu reflexión._

---
### 🧠 Rúbrica corta

| Criterio | Puntos |
|---|---|
| Reto 1: actas por mesa y suma final (idea de *map*, *combiner* y *reduce*) | 35 |
| Reto 2: copias en varias casas, falla de una casa y re-replicación | 40 |
| Reto 3: reflexión con tus cifras | 25 |
| **Total** | **100** |

**Checklist:** ✅ ejecuté *Run All* sin errores · ✅ no quedan `...` · ✅ las dos autoverificaciones imprimen ✅ · ✅ escribí mi reflexión.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>